# Chapter 2 — Step Zero: inventário de dados e viabilidade da comparação

**Objetivo deste notebook (passo zero):** antes de extrair/comparar MAF, fazer o **inventário** do que temos para a comparação
**ZNF175: v1 (Park / Freeze One) × v2 (Hui / Release 2020 v2.0)** e decidir **o que é viável** (em especial: dá para fazer concordância por-amostra?).

> Este notebook **não** extrai variantes do v1 nem calcula MAF final — isso vem nos notebooks `01` (extração v1) e `02` (comparação). Aqui é só reconhecimento.

| | v1 (Park) | v2 (Hui) |
|---|---|---|
| Dado | Freeze One | Release 2020 v2.0 (GL) |
| N | 11.451 | 43.731 |
| Build | GRCh38 | GRCh38 |

Referência: [`../proposal_znf175_maf_across_pmbb_versions.md`](../proposal_znf175_maf_across_pmbb_versions.md).

In [1]:
# --- Setup: paths and helpers ---
import gzip
from pathlib import Path
import pandas as pd

BASE    = Path("/project/hall/analysis/hearing-loss-genomics")
RESULTS = BASE / "analysis/chapter_2/results/00"
RESULTS.mkdir(parents=True, exist_ok=True)

# v1 = Park / Freeze One (GRCh38) — genome-wide PLINK .fam
V1_DIR = Path("/static/PMBB/PMBB_Freeze17/genotype/exome/all_variants")
V1_FAM = V1_DIR / "UPENN_Freeze_One_GRCh38.GL.pVCF.biallelic.fam"

# v2 = Hui / Release 2020 v2.0 — Daniel's pre-extracted chr19 .fam (full cohort)
V2_FAM = BASE / "data/PMBB_Exome/genotypes/allIndvs_chr19.fam.gz"

# Daniel's ZNF175 assets (all derived from v2 genotypes)
ZDIR = BASE / "data/PMBB_Exome/ZNF175"

print("pandas:", pd.__version__)
print("results dir:", RESULTS)
assert V1_FAM.exists() and V2_FAM.exists() and ZDIR.exists(), "check paths"
print("paths OK")

pandas: 3.0.2
results dir: /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2/results/00
paths OK


## 1. Locus do ZNF175 (GRCh38)

Precisamos da janela genômica do gene para depois recortar (`tabix`) as variantes nos dois pVCFs.
Tiramos isso da anotação que o Daniel já gerou (`ZNF175_annot_genes_full.txt.gz`), que tem uma linha por variante
anotada ao gene, com posição e função (`Func.refGene`, `ExonicFunc.refGene`), REVEL, gnomAD, ClinVar e contagens de genótipo.

In [2]:
# --- Load Daniel's ZNF175 annotation (one row per variant) ---
annot = pd.read_csv(ZDIR / "ZNF175_annot_genes_full.txt.gz", sep="\t", dtype=str)
print("total annotated rows:", len(annot))
print("n columns:", annot.shape[1])

# keep variants annotated to ZNF175 (refGene)
znf = annot[annot["Gene.refGene"].str.contains("ZNF175", na=False)].copy()
znf["Start"] = pd.to_numeric(znf["Start"], errors="coerce")
znf["End"]   = pd.to_numeric(znf["End"],   errors="coerce")

lo, hi = int(znf["Start"].min()), int(znf["End"].max())
print(f"\nZNF175 variants in annotation: {len(znf)}")
print(f"GRCh38 span (from variant positions): chr19:{lo:,}-{hi:,}")
print("NOTE: this is the span of *observed variants*, a proxy for the gene window.")
print("For tabix extraction we'll pad it (e.g. +/- 5 kb) to be safe.")

# save a padded region for downstream tabix
pad = 5000
region = f"chr19:{lo-pad}-{hi+pad}"
(RESULTS / "znf175_locus_grch38.txt").write_text(
    f"# ZNF175 GRCh38 — variant span chr19:{lo}-{hi}; padded region below (+/-{pad} bp)\n{region}\n"
)
print("padded tabix region ->", region, "(saved to results/znf175_locus_grch38.txt)")

total annotated rows: 380
n columns: 104

ZNF175 variants in annotation: 380
GRCh38 span (from variant positions): chr19:51,573,230-51,588,560
NOTE: this is the span of *observed variants*, a proxy for the gene window.
For tabix extraction we'll pad it (e.g. +/- 5 kb) to be safe.
padded tabix region -> chr19:51568230-51593560 (saved to results/znf175_locus_grch38.txt)


## 2. Sets de variantes do ZNF175 no v2 (já extraídos pelo Daniel)

O Daniel deixou **vários** extratos do ZNF175 — e eles **diferem na contagem** conforme o filtro aplicado.
Isso já ilustra o tema central do capítulo: **o conjunto de variantes muda conforme o critério**.

- `standard` — extrato direto do allIndvs (v2)
- `Joe's` — set do Joe Park (note IDs multialélicos combinados, ex. `...;...`)
- `maf.001_noRels_keepHLcases` — filtrado: MAF<0,1%, sem aparentados, mantendo casos de HL
- `inclMultAllelic_noMAFfilter` — inclui multialélicos, sem filtro de MAF (o mais completo)

In [3]:
# --- Compare Daniel's v2 ZNF175 variant sets ---
def read_bim(path):
    return pd.read_csv(path, sep="\t", header=None,
                       names=["chr","id","cm","pos","a1","a2"], dtype=str)

sets = {
    "standard (allIndvs)":          "ZNF175_allIndvs_chr19.bim.gz",
    "Joe's":                        "ZNF175_allIndvs_chr19_Joes.bim.gz",
    "maf.001_noRels_keepHLcases":   "ZNF175_allIndvs_chr19_maf.001_noRels_keepHLcases.bim.gz",
    "inclMultAllelic_noMAFfilter":  "ZNF175_allIndvs_inclMultAllelic_noMAFfilter.bim.gz",
}
bims = {name: read_bim(ZDIR / f) for name, f in sets.items()}
summary = pd.DataFrame([{"set": n, "n_variants": len(b)} for n, b in bims.items()])
print(summary.to_string(index=False))
summary.to_csv(RESULTS / "v2_znf175_variant_sets.csv", index=False)

# positions present in the most complete set
full = bims["inclMultAllelic_noMAFfilter"]
print("\nposition range (inclMultAllelic):",
      f"chr19:{full['pos'].astype(int).min():,}-{full['pos'].astype(int).max():,}")

                        set  n_variants
        standard (allIndvs)          23
                      Joe's           9
 maf.001_noRels_keepHLcases          22
inclMultAllelic_noMAFfilter          26

position range (inclMultAllelic): chr19:51,573,356-51,588,428


## 3. Anotação funcional: o que classifica pLOF e MAF

A anotação tem as colunas que precisaremos para (a) marcar **pLOF / missense** e (b) calcular **MAF**.
Aqui só **mostramos** essas colunas e contamos **portadores** — o MAF por ancestralidade fica para o notebook de comparação
(precisa do AN por ancestralidade, que não está nesta tabela).

Colunas-chave:
- `Func.refGene`, `ExonicFunc.refGene` → tipo de variante (UTR5, exonic, splicing, frameshift, stopgain…)
- `REVEL_score` → patogenicidade de missense
- `gnomAD_exome_*` → frequência de referência por população
- `CLNSIG` → ClinVar
- `HET_REF_ALT_CTS`, `TWO_ALT_GENO_CTS`, `MISSING_CT` → contagens de genótipo (na coorte anotada)

In [4]:
# --- Functional view + carrier counts (v2) ---
keep = ["ID","Start","Func.refGene","ExonicFunc.refGene","REVEL_score",
        "gnomAD_exome_ALL","gnomAD_exome_AFR","gnomAD_exome_NFE",
        "CLNSIG","avsnp150","HET_REF_ALT_CTS","TWO_ALT_GENO_CTS","MISSING_CT"]
view = znf[keep].copy()
for c in ["HET_REF_ALT_CTS","TWO_ALT_GENO_CTS","MISSING_CT"]:
    view[c] = pd.to_numeric(view[c], errors="coerce").fillna(0).astype(int)

# carriers = HET + HOM_ALT ; alt allele count = HET + 2*HOM_ALT
view["carriers"] = view["HET_REF_ALT_CTS"] + view["TWO_ALT_GENO_CTS"]
view["ac_alt"]   = view["HET_REF_ALT_CTS"] + 2*view["TWO_ALT_GENO_CTS"]

# breakdown of functional classes
print("ExonicFunc.refGene counts:")
print(znf["ExonicFunc.refGene"].value_counts(dropna=False).to_string())
print("\nFunc.refGene counts:")
print(znf["Func.refGene"].value_counts(dropna=False).to_string())

print("\nTop variants by carrier count (v2 annotated cohort):")
print(view.sort_values("carriers", ascending=False)
          .head(12)[["ID","ExonicFunc.refGene","REVEL_score","gnomAD_exome_ALL","carriers","ac_alt"]]
          .to_string(index=False))

view.to_csv(RESULTS / "v2_znf175_annotated.csv", index=False)
print("\nsaved -> results/v2_znf175_annotated.csv")
print("NB: MAF (by ancestry) NOT computed here — needs AN per ancestry (next notebook).")

ExonicFunc.refGene counts:
ExonicFunc.refGene
nonsynonymous SNV          168
.                          129
synonymous SNV              60
frameshift substitution     12
stopgain                    10
stoploss                     1

Func.refGene counts:
Func.refGene
exonic      251
intronic     95
UTR5         17
UTR3         17

Top variants by carrier count (v2 annotated cohort):
             ID ExonicFunc.refGene REVEL_score gnomAD_exome_ALL  carriers  ac_alt
19:51586847:T:C     synonymous SNV           .           0.2026     17150   19623
19:51586553:T:C                  .           .                .     16359   18378
19:51588529:A:C                  .           .                .     10366   11209
19:51581842:G:A     synonymous SNV           .           0.0767      7533    7969
19:51586761:C:T  nonsynonymous SNV       0.011           0.0763      7340    7774
19:51586729:C:G  nonsynonymous SNV       0.038           0.0563      7098    8410
19:51581583:C:T                  .       

## 4. IDs de amostra e viabilidade da **concordância por-amostra**

O teste **mais forte** para isolar artefato técnico é comparar o **genótipo da mesma pessoa** entre v1 e v2
(qualquer diferença = 100% pipeline). Isso exige que os IDs sejam **casáveis** entre as coortes.

Vamos inspecionar os formatos de ID dos dois `.fam` e tentar uma interseção ingênua.

In [5]:
# --- Sample-ID format check + naive intersection ---
v1_ids = pd.read_csv(V1_FAM, sep=r"\s+", header=None, usecols=[1])[1].astype(str)
v2_ids = pd.read_csv(V2_FAM, sep=r"\s+", header=None, usecols=[1])[1].astype(str)

print(f"v1 (Freeze One) N = {len(v1_ids):,} | example: {v1_ids.iloc[0]}")
print(f"v2 (Release 2.0) N = {len(v2_ids):,} | example: {v2_ids.iloc[0]}")

inter = set(v1_ids) & set(v2_ids)
print(f"\nnaive ID intersection: {len(inter):,}")
if len(inter) == 0:
    print(">>> IDs are in DIFFERENT namespaces (UPENN_* vs PMBB*).")
    print(">>> Per-sample concordance is BLOCKED until we find an ID crosswalk.")
    print(">>> Available path now: aggregate MAF comparison, stratified by ancestry.")

(RESULTS / "cohort_id_assessment.txt").write_text(
    f"v1_N={len(v1_ids)}\nv2_N={len(v2_ids)}\n"
    f"v1_example={v1_ids.iloc[0]}\nv2_example={v2_ids.iloc[0]}\n"
    f"naive_intersection={len(inter)}\n"
    f"per_sample_concordance_feasible={'YES' if len(inter)>0 else 'NO (needs ID crosswalk)'}\n"
)

v1 (Freeze One) N = 11,451 | example: UPENN_UPENN10000001_01e1e6e5-a310-45e5-a843-41aa47ef333a
v2 (Release 2.0) N = 43,731 | example: PMBB9640968538122

naive ID intersection: 0
>>> IDs are in DIFFERENT namespaces (UPENN_* vs PMBB*).
>>> Per-sample concordance is BLOCKED until we find an ID crosswalk.
>>> Available path now: aggregate MAF comparison, stratified by ancestry.


196

## 5. Veredito do passo zero e próximos passos

(As células acima salvam tudo em `analysis/chapter_2/results/`.)

In [6]:
# --- Write step-zero summary ---
summary_md = f'''# Step Zero — Summary

## Cohorts
- v1 (Park / Freeze One): N={len(v1_ids):,}, GRCh38, /static/PMBB/PMBB_Freeze17/genotype/exome/
- v2 (Hui / Release 2020 v2.0): N={len(v2_ids):,}, GRCh38, data/pmbb_v2/Exome/pVCF/GL_by_chrom/

## ZNF175 locus (GRCh38)
- variant span: chr19:{lo:,}-{hi:,}  (padded tabix region in znf175_locus_grch38.txt)

## v2 variant sets already extracted by Daniel (counts differ by filter)
{summary.to_string(index=False)}

## Per-sample concordance feasibility
- naive ID intersection (v1 vs v2): {len(inter):,}
- verdict: {"FEASIBLE" if len(inter)>0 else "BLOCKED — IDs in different namespaces (UPENN_* vs PMBB*); need an ID crosswalk"}
- fallback available now: aggregate MAF comparison, stratified by ancestry

## Next steps
- NB 01: extract ZNF175 from v1 (Freeze One) via tabix region on the GRCh38 pVCF
- NB 02: harmonize variants (normalize, match by chr:pos:ref:alt) and compare MAF v1 vs v2 (by ancestry)
- Side-quest: locate a UPENN_* <-> PMBB ID crosswalk to unlock per-sample concordance
'''
(RESULTS / "step_zero_summary.md").write_text(summary_md)
print(summary_md)

# Step Zero — Summary

## Cohorts
- v1 (Park / Freeze One): N=11,451, GRCh38, /static/PMBB/PMBB_Freeze17/genotype/exome/
- v2 (Hui / Release 2020 v2.0): N=43,731, GRCh38, data/pmbb_v2/Exome/pVCF/GL_by_chrom/

## ZNF175 locus (GRCh38)
- variant span: chr19:51,573,230-51,588,560  (padded tabix region in znf175_locus_grch38.txt)

## v2 variant sets already extracted by Daniel (counts differ by filter)
                        set  n_variants
        standard (allIndvs)          23
                      Joe's           9
 maf.001_noRels_keepHLcases          22
inclMultAllelic_noMAFfilter          26

## Per-sample concordance feasibility
- naive ID intersection (v1 vs v2): 0
- verdict: BLOCKED — IDs in different namespaces (UPENN_* vs PMBB*); need an ID crosswalk
- fallback available now: aggregate MAF comparison, stratified by ancestry

## Next steps
- NB 01: extract ZNF175 from v1 (Freeze One) via tabix region on the GRCh38 pVCF
- NB 02: harmonize variants (normalize, match by chr:pos:ref